# Playground: Land Subsidence Susceptibility Mapping (Tulare-Corcoran, San Joaquin Valley)

Module 7 warm-up for Milestone 3. This notebook loads the project's grid-cell dataset and takes a first look at it.

**Dataset:** `data/subsidence_dataset.csv`, one row per ~500 m grid cell, built by `build_dataset.py` from real public sources (see README for sources, retrieval date and licenses).
**Status:** the InSAR displacement label is not included yet (the DWR InSAR rasters are hosted on a Box page rather than a direct download, so I am working from the point data instead). This notebook covers the predictor features.

## 1. Setup

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt
pd.set_option("display.max_columns", None)
plt.rcParams["figure.figsize"] = (8, 4)

## 2. Retrieve the dataset
The file lives in this repository's `data/` folder (about 1.6 MB, built from public sources). It is loaded by a relative path, so the notebook runs top to bottom on any machine with the repo checked out. To rebuild it from the original sources, run `python build_dataset.py`.

In [ ]:
DATA_PATH = "data/subsidence_dataset.csv"
df = pd.read_csv(DATA_PATH)
print(f"Loaded {DATA_PATH!r}")

## 3. Basic dataset information

In [ ]:
print(f"Rows (grid cells): {df.shape[0]}")
print(f"Columns (fields):  {df.shape[1]}")

In [ ]:
df.dtypes

In [ ]:
df.head(5)

In [ ]:
df.describe().T

## 4. Look while you're here: data quality checks
A first look, not a cleaning pass. Problems without a fix yet are recorded here and feed Section 6.4.

In [ ]:
# Missing values per column
miss = df.isna().sum()
pd.DataFrame({"missing": miss, "pct": (miss / len(df) * 100).round(1)}).query("missing > 0").sort_values("missing", ascending=False)

In [ ]:
# Duplicate grid cells
print("Duplicate grid_id values:", df["grid_id"].duplicated().sum())
print("Fully duplicated rows:  ", df.duplicated().sum())

In [ ]:
# Land use: what covers the area, and how much of it is cropland
print(df["land_use_name_last_year"].value_counts().head(12))
df["land_use_name_last_year"].value_counts().head(12).plot(kind="barh", color="#2E7D32")
plt.title("Land use in the last predictor year"); plt.gca().invert_yaxis(); plt.tight_layout(); plt.show()

In [ ]:
# Groundwater coverage: every cell is filled from its NEAREST well, so distance shows how far
# a value really is from a measurement.
print(df["dist_to_nearest_well_m"].describe())
df["dist_to_nearest_well_m"].plot(kind="hist", bins=40, color="#2F5496", edgecolor="black")
plt.xlabel("Distance to nearest well (m)"); plt.title("Groundwater data coverage"); plt.tight_layout(); plt.show()

In [ ]:
# Where is soil data missing? (checks whether gaps are clustered in space)
miss_soil = df["clay_pct"].isna()
plt.scatter(df.longitude, df.latitude, s=4, c=np.where(miss_soil, "#B02A2A", "#CCCCCC"))
plt.title("Cells with missing soil data (red)"); plt.xlabel("longitude"); plt.ylabel("latitude"); plt.show()

In [ ]:
# Plausible ranges
cols = ["gw_decline_m_per_yr_2016_2019", "gw_mean_depth_m", "elevation_m", "slope_deg", "clay_pct", "sand_pct", "silt_pct"]
df[cols].agg(["min", "max"]).T

### Findings from this first look

- **Soil gaps:** about 6% of cells have no soil map unit and about 8% are missing the soil properties (clay, sand, silt, organic matter and others). I have no fix for this yet. The map above shows where they fall; I still need to decide whether to drop, impute or flag these cells, and this ties to the "no-data" state in my Milestone 2 plan.
- **Groundwater is interpolated by nearest well:** no cell is missing a groundwater value, but that is only because each cell takes its nearest well's value. The distance histogram shows how far many cells are from a real measurement. Values for far-away cells are weakly supported.
- **No label yet:** the InSAR displacement and the susceptibility class are not in this file. Without them I cannot check class balance. This is the main gap before modeling.
- **Spatial coverage:** the dataset covers the whole bounding box, but the groundwater source was limited to a sample of wells in Tulare and Kings counties, so coverage is uneven.
- **My own observation from the outputs above:** (add one sentence here after you run the cells, for example the most common land use, or how far the typical cell is from a well).

## 5. Data provenance
See `README.md`. Groundwater: CA DWR periodic groundwater level measurements. Land use: USDA NASS Cropland Data Layer. Soil: USDA NRCS Soil Data Access (SSURGO). Elevation: USGS 3DEP. Retrieval date and license text are recorded there.